download_datasets.py
---------------------
Script tải các dataset Kaggle (Hotel Booking Demand, Instacart Market Basket
Analysis, US Accidents) về máy cục bộ bằng kagglehub.

Cách dùng: Chạy script: chạy trực tiếp trong Jupyter Notebook

Dữ liệu sẽ được tải vào cache local (~/.cache/kagglehub/) và KHÔNG được
đưa vào repo Git (xem file .gitignore đi kèm).

In [8]:
import os
import sys
import subprocess
import shutil

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
RAW_DIR = os.environ.get("RAW_DATA_DIR", SCRIPT_DIR)
os.makedirs(RAW_DIR, exist_ok=True)


def ensure_package(package: str, import_name: str = None) -> None:
    """Tự động cài đặt package bằng pip nếu chưa có sẵn."""
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        print(f"📦 Chưa có '{package}', đang cài đặt...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )
        print(f"✅ Đã cài xong '{package}'")

ensure_package("kagglehub")


In [9]:
import kagglehub

DATASETS = {
    "D1_hotel_booking": "jessemostipak/hotel-booking-demand",
    "D2_instacart": "psparks/instacart-market-basket-analysis",
    "D3_us_accidents": "sobhanmoosavi/us-accidents",
}

def download_all(datasets: dict) -> dict:
    """
    Tải toàn bộ dataset trong `datasets`, copy vào thư mục RAW_DIR/<tên>,
    trả về dict {tên: đường dẫn trong thư mục raw}.
    """
    final_paths = {}

    for name, dataset_id in datasets.items():
        dest_dir = os.path.join(RAW_DIR, name)

        if os.path.isdir(dest_dir) and os.listdir(dest_dir):
            print(f"⏭️  Bỏ qua {name}: đã tồn tại tại {dest_dir}")
            final_paths[name] = dest_dir
            print("-" * 60)
            continue
 
        print(f"⬇️  Đang tải {name} ({dataset_id}) ...")
        try:
            # kagglehub tải/caching về ~/.cache/kagglehub/ trước
            cache_path = kagglehub.dataset_download(dataset_id)
 
            # Copy từ cache sang thư mục raw đã có sẵn
            shutil.copytree(cache_path, dest_dir, dirs_exist_ok=True)
 
            final_paths[name] = dest_dir
            print(f"✅ Xong: {name} -> {dest_dir}")
        except Exception as e:
            print(f"❌ Lỗi khi tải {name}: {e}")
        print("-" * 60)
 
    return final_paths
 
 
def list_dataset_files(downloaded_paths: dict) -> None:
    """In ra danh sách file và dung lượng của từng dataset đã tải."""
    for name, path in downloaded_paths.items():
        print(f"\n📁 {name}  ({path})")
        if not os.path.isdir(path):
            print("   (không tìm thấy thư mục)")
            continue
        for f in sorted(os.listdir(path)):
            full_path = os.path.join(path, f)
            if os.path.isfile(full_path):
                size_mb = os.path.getsize(full_path) / (1024 * 1024)
                print(f"   - {f} ({size_mb:.2f} MB)")
            else:
                print(f"   - {f}/ (thư mục con)")
 
 
def main():
    print("=" * 60)
    print("TẢI DATASET TỪ KAGGLE")
    print(f"Thư mục lưu trữ: {RAW_DIR}")
    print("=" * 60)
 
    downloaded_paths = download_all(DATASETS)
 
    print("\n📂 Tổng hợp đường dẫn:")
    for name, path in downloaded_paths.items():
        print(f"   {name}: {path}")
 
    print("\n🔍 Chi tiết nội dung từng dataset:")
    list_dataset_files(downloaded_paths)
 
    return downloaded_paths
 
 
if __name__ == "__main__":
    paths = main()


TẢI DATASET TỪ KAGGLE
Thư mục lưu trữ: d:\data-mining\data\raw
⬇️  Đang tải D1_hotel_booking (jessemostipak/hotel-booking-demand) ...


100%|██████████| 1.25M/1.25M [00:01<00:00, 959kB/s]

Extracting files...


✅ Xong: D1_hotel_booking -> d:\data-mining\data\raw\D1_hotel_booking
------------------------------------------------------------
⬇️  Đang tải D2_instacart (psparks/instacart-market-basket-analysis) ...


100%|██████████| 197M/197M [00:12<00:00, 17.3MB/s] 

Extracting files...


✅ Xong: D2_instacart -> d:\data-mining\data\raw\D2_instacart
------------------------------------------------------------
⬇️  Đang tải D3_us_accidents (sobhanmoosavi/us-accidents) ...


100%|██████████| 653M/653M [00:39<00:00, 17.4MB/s] 

Extracting files...


✅ Xong: D3_us_accidents -> d:\data-mining\data\raw\D3_us_accidents
------------------------------------------------------------

📂 Tổng hợp đường dẫn:
   D1_hotel_booking: d:\data-mining\data\raw\D1_hotel_booking
   D2_instacart: d:\data-mining\data\raw\D2_instacart
   D3_us_accidents: d:\data-mining\data\raw\D3_us_accidents

🔍 Chi tiết nội dung từng dataset:

📁 D1_hotel_booking  (d:\data-mining\data\raw\D1_hotel_booking)
   - hotel_bookings.csv (16.07 MB)

📁 D2_instacart  (d:\data-mining\data\raw\D2_instacart)
   - aisles.csv (0.00 MB)
   - departments.csv (0.00 MB)
   - order_products__prior.csv (550.80 MB)
   - order_products__train.csv (23.54 MB)
   - orders.csv (103.92 MB)
   - products.csv (2.07 MB)

📁 D3_us_accidents  (d:\data-mining\data\raw\D3_us_accidents)
   - US_Accidents_March23.csv (2916.51 MB)
